# Taller 3.3 — Cálculo y Visualización de Normales

## Objetivo
Calcular normales de **caras** y **vértices** desde cero usando álgebra vectorial,
comparar *flat shading* vs *smooth shading*, y validar los resultados contra la
biblioteca `trimesh`.

## 1. Instalación Automática de Dependencias

In [ ]:
import subprocess
import sys

libraries = ['numpy', 'matplotlib', 'trimesh', 'pillow', 'imageio']

for lib in libraries:
    pkg = lib if lib != 'pillow' else 'PIL'
    try:
        __import__(pkg)
        print(f'✓ {lib} ya está instalado')
    except ImportError:
        print(f'Installing {lib}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', lib, '-q'])
        print(f'✓ {lib} instalado correctamente')

print('\n✅ Todas las dependencias están listas!')

## 2. Importar Librerías

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import trimesh
import os
import warnings
warnings.filterwarnings('ignore')

os.makedirs('../media', exist_ok=True)

print('✓ Librerías importadas correctamente')

## 3. Crear Malla 3D de Referencia

Usamos `trimesh` para crear una esfera icosahédrica con bajo número de caras,
ideal para visualizar normales individuales.

In [ ]:
# Crear una esfera icosahédrica de baja resolución
mesh = trimesh.creation.icosphere(subdivisions=2, radius=1.0)

vertices = np.array(mesh.vertices)   # (V, 3)
faces    = np.array(mesh.faces)      # (F, 3)  índices de vértices

print(f'Vértices : {len(vertices)}')
print(f'Caras    : {len(faces)}')
print(f'Shape vértices : {vertices.shape}')
print(f'Shape caras    : {faces.shape}')

## 4. Cálculo Manual de Normales de Caras

Para cada triángulo $(v_0, v_1, v_2)$:
$$\mathbf{n}_{cara} = \frac{(v_1 - v_0) \times (v_2 - v_0)}{\|(v_1 - v_0) \times (v_2 - v_0)\|}$$

In [ ]:
def compute_face_normals(vertices, faces):
    """Calcula la normal de cada cara (triángulo) usando producto cruz."""
    v0 = vertices[faces[:, 0]]   # (F, 3)
    v1 = vertices[faces[:, 1]]
    v2 = vertices[faces[:, 2]]

    edge1 = v1 - v0             # (F, 3)
    edge2 = v2 - v0

    normals = np.cross(edge1, edge2)   # (F, 3)  — producto cruz

    # Normalizar a magnitud unitaria
    magnitudes = np.linalg.norm(normals, axis=1, keepdims=True)
    magnitudes = np.where(magnitudes == 0, 1e-10, magnitudes)   # evitar div/0
    normals = normals / magnitudes

    return normals   # (F, 3)


face_normals_manual = compute_face_normals(vertices, faces)

# Verificar que apunten hacia afuera (dot con centroide de cara > 0)
centroids = vertices[faces].mean(axis=1)   # (F, 3)
dot_products = (face_normals_manual * centroids).sum(axis=1)   # (F,)
outward_pct = (dot_products > 0).mean() * 100

print(f'Normales de cara calculadas: {face_normals_manual.shape}')
print(f'Magnitud media              : {np.linalg.norm(face_normals_manual, axis=1).mean():.6f}')
print(f'Apuntan hacia afuera       : {outward_pct:.1f}%')

# Comparar contra trimesh
face_normals_trimesh = np.array(mesh.face_normals)
max_diff = np.abs(face_normals_manual - face_normals_trimesh).max()
print(f'Diferencia máxima vs trimesh: {max_diff:.2e}')

## 5. Cálculo Manual de Normales de Vértices

La normal de cada vértice es el promedio normalizado de las normales de las caras
que lo comparten:
$$\mathbf{n}_{vértice} = \frac{\sum_{f \ni v} \mathbf{n}_f}{\left\|\sum_{f \ni v} \mathbf{n}_f\right\|}$$

In [ ]:
def compute_vertex_normals(vertices, faces, face_normals):
    """Promedia normales de caras adyacentes para obtener la normal de cada vértice."""
    n_vertices = len(vertices)
    vertex_normals = np.zeros((n_vertices, 3))

    # Acumular normal de cara en cada uno de sus 3 vértices
    for fi, (i0, i1, i2) in enumerate(faces):
        vertex_normals[i0] += face_normals[fi]
        vertex_normals[i1] += face_normals[fi]
        vertex_normals[i2] += face_normals[fi]

    # Normalizar
    magnitudes = np.linalg.norm(vertex_normals, axis=1, keepdims=True)
    magnitudes = np.where(magnitudes == 0, 1e-10, magnitudes)
    vertex_normals = vertex_normals / magnitudes

    return vertex_normals   # (V, 3)


vertex_normals_manual = compute_vertex_normals(vertices, faces, face_normals_manual)

# Comparar contra trimesh
vertex_normals_trimesh = np.array(mesh.vertex_normals)
max_diff_v = np.abs(np.abs(vertex_normals_manual) - np.abs(vertex_normals_trimesh)).max()

print(f'Normales de vértice calculadas : {vertex_normals_manual.shape}')
print(f'Magnitud media                 : {np.linalg.norm(vertex_normals_manual, axis=1).mean():.6f}')
print(f'Diferencia máxima vs trimesh   : {max_diff_v:.2e}')

## 6. Visualización de Normales de Cara y Vértice

Dibujamos las normales como flechas (quivers) sobre el modelo.

In [ ]:
ARROW_LEN = 0.18

fig = plt.figure(figsize=(14, 6))

# ── Normales de cara ──────────────────────────────────────────────────────────
ax1 = fig.add_subplot(121, projection='3d')

poly_collection = Poly3DCollection(
    vertices[faces], alpha=0.15,
    facecolor='steelblue', edgecolor='white', linewidth=0.2
)
ax1.add_collection3d(poly_collection)

origins = centroids
ax1.quiver(
    origins[:, 0], origins[:, 1], origins[:, 2],
    face_normals_manual[:, 0] * ARROW_LEN,
    face_normals_manual[:, 1] * ARROW_LEN,
    face_normals_manual[:, 2] * ARROW_LEN,
    color='red', linewidth=0.7, arrow_length_ratio=0.3
)
ax1.set_title('Normales de Cara (producto cruz)', fontsize=11)
ax1.set_xlim(-1.3, 1.3); ax1.set_ylim(-1.3, 1.3); ax1.set_zlim(-1.3, 1.3)
ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')
ax1.set_facecolor('#0a0a1a')
fig.patch.set_facecolor('#0a0a1a')

# ── Normales de vértice ───────────────────────────────────────────────────────
ax2 = fig.add_subplot(122, projection='3d')

poly_collection2 = Poly3DCollection(
    vertices[faces], alpha=0.15,
    facecolor='darkorange', edgecolor='white', linewidth=0.2
)
ax2.add_collection3d(poly_collection2)

# Dibujar subset para no saturar
step = max(1, len(vertices) // 80)
idxs = np.arange(0, len(vertices), step)
ax2.quiver(
    vertices[idxs, 0], vertices[idxs, 1], vertices[idxs, 2],
    vertex_normals_manual[idxs, 0] * ARROW_LEN,
    vertex_normals_manual[idxs, 1] * ARROW_LEN,
    vertex_normals_manual[idxs, 2] * ARROW_LEN,
    color='lime', linewidth=0.7, arrow_length_ratio=0.3
)
ax2.set_title('Normales de Vértice (promedio de caras)', fontsize=11)
ax2.set_xlim(-1.3, 1.3); ax2.set_ylim(-1.3, 1.3); ax2.set_zlim(-1.3, 1.3)
ax2.set_xlabel('X'); ax2.set_ylabel('Y'); ax2.set_zlabel('Z')
ax2.set_facecolor('#0a0a1a')

for ax in [ax1, ax2]:
    ax.tick_params(colors='white')
    ax.xaxis.label.set_color('white')
    ax.yaxis.label.set_color('white')
    ax.zaxis.label.set_color('white')
    ax.title.set_color('white')

plt.tight_layout()
plt.savefig('../media/normales_cara_vertice.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('✓ Guardado: ../media/normales_cara_vertice.png')

## 7. Flat Shading vs Smooth Shading

Simulamos iluminación Lambert con los dos tipos de normales para mostrar la diferencia visual.

In [ ]:
LIGHT_DIR = np.array([1.0, 1.5, 2.0])
LIGHT_DIR = LIGHT_DIR / np.linalg.norm(LIGHT_DIR)

BASE_COLOR_FLAT   = np.array([0.3, 0.6, 1.0])   # azul
BASE_COLOR_SMOOTH = np.array([1.0, 0.5, 0.2])   # naranja

def lambert_face_color(normals, base_color, ambient=0.15):
    """Calcula color Lambert por normal (cara o vértice)."""
    diff = np.clip((normals @ LIGHT_DIR), 0, 1)
    return np.clip(base_color * (ambient + (1 - ambient) * diff[:, np.newaxis]), 0, 1)

# Color por cara (flat: una normal por triángulo)
face_colors_flat   = lambert_face_color(face_normals_manual, BASE_COLOR_FLAT)

# Color por vértice promediado por cara (smooth)
vertex_colors_avg  = vertex_normals_manual[faces].mean(axis=1)   # promedio por cara
face_colors_smooth = lambert_face_color(vertex_colors_avg, BASE_COLOR_SMOOTH)

fig = plt.figure(figsize=(14, 6))
fig.patch.set_facecolor('#0a0a1a')

for col, (title, fcolors) in enumerate([
    ('Flat Shading\n(normal por cara)', face_colors_flat),
    ('Smooth Shading\n(normal por vértice)', face_colors_smooth),
]):
    ax = fig.add_subplot(1, 2, col + 1, projection='3d')
    rgba = np.hstack([fcolors, np.ones((len(fcolors), 1))])   # añadir alpha
    poly = Poly3DCollection(vertices[faces], alpha=1.0, facecolor=rgba, edgecolor='none')
    ax.add_collection3d(poly)
    ax.set_xlim(-1.2, 1.2); ax.set_ylim(-1.2, 1.2); ax.set_zlim(-1.2, 1.2)
    ax.set_title(title, fontsize=12, color='white')
    ax.set_facecolor('#0a0a1a')
    ax.set_axis_off()

plt.suptitle('Flat vs Smooth Shading — Iluminación Lambert', color='white', fontsize=13)
plt.tight_layout()
plt.savefig('../media/flat_vs_smooth_shading.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('✓ Guardado: ../media/flat_vs_smooth_shading.png')

## 8. Validación de Normales

- **Magnitud ≈ 1**: todas las normales deben ser unitarias.
- **Orientación consistente**: producto punto con el centroide debe ser positivo para una esfera.
- **Detección y corrección** de normales invertidas.

In [ ]:
print('=== Validación de normales de CARA ===')
mags_face = np.linalg.norm(face_normals_manual, axis=1)
print(f'  Magnitud mínima : {mags_face.min():.6f}')
print(f'  Magnitud máxima : {mags_face.max():.6f}')
print(f'  Magnitud media  : {mags_face.mean():.6f}')
all_unit_face = np.allclose(mags_face, 1.0, atol=1e-5)
print(f'  Todas unitarias (atol=1e-5) : {all_unit_face}')

dot_face = (face_normals_manual * centroids).sum(axis=1)
n_inverted_face = (dot_face < 0).sum()
print(f'  Normales invertidas : {n_inverted_face}')

print()
print('=== Validación de normales de VÉRTICE ===')
mags_vert = np.linalg.norm(vertex_normals_manual, axis=1)
print(f'  Magnitud mínima : {mags_vert.min():.6f}')
print(f'  Magnitud máxima : {mags_vert.max():.6f}')
print(f'  Magnitud media  : {mags_vert.mean():.6f}')
all_unit_vert = np.allclose(mags_vert, 1.0, atol=1e-5)
print(f'  Todas unitarias (atol=1e-5) : {all_unit_vert}')

dot_vert = (vertex_normals_manual * vertices).sum(axis=1)   # para esfera: centroide = vértice
n_inverted_vert = (dot_vert < 0).sum()
print(f'  Normales invertidas : {n_inverted_vert}')

print()
print('=== Corrección de normales invertidas (si las hubiera) ===')
corrected = face_normals_manual.copy()
mask = dot_face < 0
corrected[mask] *= -1
print(f'  Normales corregidas : {mask.sum()}')
print(f'  Orientación post-corrección: {((corrected * centroids).sum(axis=1) > 0).mean()*100:.1f}% apuntando afuera')

## 9. Visualización del Mapa de Normales (Normal Map)

Mapeamos las componentes XYZ de la normal de cada cara a colores RGB.

In [ ]:
# Color de cada cara: normal_xyz * 0.5 + 0.5  → [0, 1]
normal_colors = face_normals_manual * 0.5 + 0.5   # (F, 3)

fig = plt.figure(figsize=(10, 8))
fig.patch.set_facecolor('#0a0a1a')
ax = fig.add_subplot(111, projection='3d')

rgba = np.hstack([normal_colors, np.ones((len(normal_colors), 1))])
poly = Poly3DCollection(vertices[faces], alpha=1.0, facecolor=rgba, edgecolor='none')
ax.add_collection3d(poly)

ax.set_xlim(-1.2, 1.2); ax.set_ylim(-1.2, 1.2); ax.set_zlim(-1.2, 1.2)
ax.set_title('Mapa de Normales\n(XYZ → RGB)', fontsize=13, color='white')
ax.set_facecolor('#0a0a1a')
ax.set_axis_off()

plt.tight_layout()
plt.savefig('../media/normal_map.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('✓ Guardado: ../media/normal_map.png')

## 10. Resumen

In [ ]:
print('=' * 50)
print('RESUMEN — Taller 3.3')
print('=' * 50)
print(f'Malla           : icosphere  ({len(vertices)} vértices, {len(faces)} caras)')
print(f'Normal cara     : producto cruz de aristas')
print(f'Normal vértice  : promedio ponderado de caras adyacentes')
print(f'Error vs trimesh (cara)    : {max_diff:.2e}')
print(f'Error vs trimesh (vértice) : {max_diff_v:.2e}')
print(f'Normales invertidas detectadas (cara)    : {n_inverted_face}')
print(f'Normales invertidas detectadas (vértice) : {n_inverted_vert}')
print()
print('Archivos generados:')
for f in ['normales_cara_vertice.png', 'flat_vs_smooth_shading.png', 'normal_map.png']:
    path = f'../media/{f}'
    exists = os.path.exists(path)
    print(f'  {"✓" if exists else "✗"} {path}')